In [92]:
def parseAuthors(authorString):
    # Parsing the author string, which doesn't have any consistency (yippee!)
    authorsAndSplit = authorString.split(" and ")
    authors = []

    for author in authorsAndSplit:
        # Doing weird gymnastics to handle Oxford commas that were malformed due to the " and " split
        authorsCommaSplit = author.replace(", ", "|").replace(",", "").split("|")
        authors += authorsCommaSplit

    return authors

def toBibtexString(data):
    citationName = data["authors"][0].split(" ")[-1].replace("\'", "-") + data["year"]

    return f'''
    @inproceedings{{{citationName},
        author = {{{" and ".join(data["authors"])}}},
        title = {{{data["title"]}}},
        pages = {{{data["pages"]}}},
        booktitle = {{{data["booktitle"]}}},
        year = {{{data["year"]}}},
        publisher = {{{data["publisher"]}}},
        address = {{{data["address"]}}}
    }}
    '''

In [95]:
f = open("citations.txt", "r")
content = f.read()
f.close()

bibtexStrings = {}

year = ""
publisher = ""
address = ""
newConference = False

lines = content.split('\n')
i = 0

while i < len(lines):
    line = lines[i]

    # Handling new proceedings markers
    if len(line) > len("PROCEEDINGS") and line[0 : len("PROCEEDINGS")] == "PROCEEDINGS":
        year = line[line.index("CONFERENCE") + len("CONFERENCE") + 1 : -1]
        newConference = True
    else:
        # Handling location info underneath the PROCEEDINGS 
        if (newConference):
            locationSplit = line.split(", ", maxsplit=1)
            publisher = locationSplit[0]
            address = locationSplit[1]
            newConference = False
        else:
            # Handling weird line breaks where the citation is pushed onto a new line
            while i+1 < len(lines) and lines[i+1][0 : len("PROCEEDINGS")] != "PROCEEDINGS" and len(lines[i+1].split(" - ")) == 1:
                line += " " + lines[i+1]
                i += 1

            # Splitting entries by " - " and creating the BibTeX string using a dictionary
            fields = line.split(" - ")
            bibtexString = toBibtexString({
                "authors": parseAuthors(fields[1]),
                "title": fields[2],
                "pages": fields[0],
                "booktitle": "Proceedings of the International Computer Music Conference",
                "year": year,
                "publisher": publisher,
                "address": address
            })
            if (bibtexStrings.get(year) == None):
                bibtexStrings.update({year: [bibtexString]})
            else:
                bibtexStrings[year].append(bibtexString)

    i += 1

for year, bibs in bibtexStrings.items():
    # Skipping existing bib entries
    if year in ["1975", "1977", "1978"]:
        continue

    f = open(f"{year}.bib", "w")
    f.write("".join(bibs) if len(bibs) > 1 else bibs[0])
    f.close()

print("Done")

Done
